In [1]:
from sqlmodel import create_engine, select, Session
from experiment import Model, Result, Celltype, Dataset
import pandas as pd
import mlflow
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

/Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-02-27T09-05-08Z/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
exp_name = "Default"
benchmark_experiment = mlflow.get_experiment_by_name(exp_name)
if not benchmark_experiment:
    raise ValueError(f"Experiment '{exp_name}' not found.")
runs = mlflow.search_runs(experiment_ids=[benchmark_experiment.experiment_id])
if runs.empty:
    raise RuntimeError(f"No runs found in experiment '{exp_name}'.")
last_run_id = runs.sort_values("start_time", ascending=False)["run_id"].iloc[0]
logger.info(f"Using run_id: {last_run_id}")
logger.info("Downloading 'database.db'...")
database_path = mlflow.artifacts.download_artifacts(
    run_id=last_run_id, artifact_path="database.db"
)
logger.info(f"Local path: {database_path}")
sqlite_url = f"sqlite:///{database_path}"
engine = create_engine(sqlite_url, echo=True)
logger.info("Database engine initialized.")

INFO: Using run_id: 6754da4be975469b8f75d6638abc7500
INFO: Downloading 'database.db'...
INFO: Local path: /var/folders/qm/v_v5_1r52bx792m7x2mh177c0000gn/T/tmp6qd5xwqs/database.db
INFO: Database engine initialized.


In [9]:
engine = create_engine("sqlite:////Users/egerc/Documents/Projects/notebook_repository/notebooks/2026-02-27T09-05-08Z/mlruns/0/f3f806556030452f84cf8deec60c181c/artifacts/database.db")

In [3]:
with Session(engine) as session:
    # 1. Join Result and Celltype to get access to both matrices
    statement = select(Result, Celltype).join(Celltype)
    results = session.exec(statement).all()
    for result, celltype in results:
        actual = celltype.counts_matrix
        print(actual)

2026-03-04 13:57:18,464 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026/03/04 13:57:18 INFO sqlalchemy.engine.Engine: BEGIN (implicit)


2026-03-04 13:57:18,468 INFO sqlalchemy.engine.Engine SELECT result.id, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, celltype.id AS id_1, celltype.name, celltype.counts_matrix, celltype.dataset_id 
FROM result JOIN celltype ON celltype.id = result.celltype_id


2026/03/04 13:57:18 INFO sqlalchemy.engine.Engine: SELECT result.id, result.global_model_embedding, result.global_model_counts, result.celltype_model_embedding, result.celltype_model_counts, result.celltype_id, result.model_id, result.sample_id, celltype.id AS id_1, celltype.name, celltype.counts_matrix, celltype.dataset_id 
FROM result JOIN celltype ON celltype.id = result.celltype_id


2026-03-04 13:57:18,469 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()


2026/03/04 13:57:18 INFO sqlalchemy.engine.Engine: [generated in 0.00079s] ()


[[125.857861   109.07342529  94.1919984  ...  69.6222354   68.68821257
  101.0206789 ]
 [123.67803823 106.0765369   98.45414873 ... 118.47218073 112.03918464
  101.44534923]
 [108.38509668  89.95510897 142.50735388 ...  88.95527589  66.80467013
   77.51965333]
 ...
 [ 67.88384216  66.92922246 102.29091601 ... 148.50105362 102.04673994
   69.03001601]
 [116.40463794  79.2611102   95.36432848 ...  69.94393909 104.80255676
  126.71105686]
 [103.15977043 110.61043394  81.17933391 ...  95.75333963 128.64007961
   91.02322603]]
[[125.857861   109.07342529  94.1919984  ...  69.6222354   68.68821257
  101.0206789 ]
 [123.67803823 106.0765369   98.45414873 ... 118.47218073 112.03918464
  101.44534923]
 [108.38509668  89.95510897 142.50735388 ...  88.95527589  66.80467013
   77.51965333]
 ...
 [ 67.88384216  66.92922246 102.29091601 ... 148.50105362 102.04673994
   69.03001601]
 [116.40463794  79.2611102   95.36432848 ...  69.94393909 104.80255676
  126.71105686]
 [103.15977043 110.61043394  81.

2026/03/04 13:57:18 INFO sqlalchemy.engine.Engine: ROLLBACK


In [5]:
actual.shape

(317, 500)

In [ ]:
def placeholder()

In [4]:
actual

ArrayView([[102.51460442,  97.35790273,  87.53451075, ...,  88.28942352,
             73.17560572,  71.9695957 ],
           [ 75.36283006, 113.67345166, 112.07302998, ...,  48.11814011,
            136.46495732,  80.59685596],
           [103.83899321,  77.72299631, 109.77191905, ...,  96.14349394,
             74.69416052, 122.00987104],
           ...,
           [ 82.78210809, 105.50020507,  96.7469569 , ..., 106.64036989,
             84.40316948,  79.71584715],
           [ 83.06519228, 136.58211113, 120.9374754 , ..., 106.15233411,
             79.48706224,  86.39081491],
           [121.29313861,  95.58018421,  85.61702003, ..., 100.77094282,
            106.60426802, 113.90487689]], shape=(317, 500))

In [ ]:
for result, celltype in results:
            # 2. Extract the actual matrices
            # Note: We use global_model_outputs here as an example
            predicted = result.global_model_outputs.predicted_counts
            actual = celltype.counts_matrix

AttributeError: 'bytes' object has no attribute 'predicted_counts'

In [84]:
actual = celltype.counts_matrix